In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import classification_report, confusion_matrix
from collections import Counter
from PIL import UnidentifiedImageError
import timm  # Transformer models

In [3]:
# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [4]:
# Dataset class

class ChestXRayDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform
        self.label_map = {"normal": 0, "bacterial": 1, "viral": 2}

        for label in self.label_map:
            label_dir = os.path.join(root_dir, label)
            for file in os.listdir(label_dir):
                if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.samples.append((os.path.join(label_dir, file), self.label_map[label]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):  # <- This must be inside the class, not outside
        path, label = self.samples[idx]
        try:
            image = Image.open(path).convert("RGB")
        except UnidentifiedImageError:
            print(f"Skipping corrupted file: {path}")
            return self.__getitem__((idx + 1) % len(self.samples))  # Skip corrupted image

        if self.transform:
            image = self.transform(image)
        return image, label


In [5]:
# Transforms
img_size = 224
transform_train = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

transform_val_test = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])



In [6]:
# Datasets and loaders
train_dataset = ChestXRayDataset("/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray_images/train", transform_train)
val_dataset   = ChestXRayDataset("/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray_images/val", transform_val_test)
test_dataset  = ChestXRayDataset("/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray_images/test", transform_val_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=32, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=32, num_workers=2)

In [7]:

def clean_dataset(folder):
    bad_files = []
    for root, _, files in os.walk(folder):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                full_path = os.path.join(root, file)
                try:
                    img = Image.open(full_path)
                    img.verify()
                except Exception as e:
                    print("Corrupted:", full_path)
                    bad_files.append(full_path)
    print(f"\nTotal corrupted: {len(bad_files)}")
    for f in bad_files:
        os.remove(f)

# Run this once
clean_dataset("/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray_images")


Total corrupted: 0


In [8]:
def count_dataset_classes(dataset, dataset_name):
    label_map_inv = {0: 'normal', 1: 'bacterial', 2: 'viral'}
    label_counts = Counter(label for _, label in dataset.samples)

    print(f"\n-->  {dataset_name.upper()} DATASET:")
    for label, count in sorted(label_counts.items()):
        class_name = label_map_inv.get(label, f'class_{label}')
        print(f"  {class_name}: {count} images")

# Count for each dataset
count_dataset_classes(train_dataset, "train")
count_dataset_classes(val_dataset, "val")
count_dataset_classes(test_dataset, "test")


-->  TRAIN DATASET:
  normal: 1808 images
  bacterial: 1945 images
  viral: 1827 images

-->  VAL DATASET:
  normal: 259 images
  bacterial: 278 images
  viral: 261 images

-->  TEST DATASET:
  normal: 517 images
  bacterial: 556 images
  viral: 523 images


In [14]:
# ResFormer (CNN + ViT Hybrid Model)
class ResFormer(nn.Module):
    def __init__(self, num_classes=3):
        super(ResFormer, self).__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.cnn = nn.Sequential(*list(resnet.children())[:-2])
        self.cnn_out_dim = 2048

        # Use lightweight transformer
        self.transformer = timm.create_model('vit_tiny_patch16_224', pretrained=True)
        self.transformer.head = nn.Identity()

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(self.cnn_out_dim + 192, num_classes)  # 2048 (CNN) + 192 (ViT)

    def forward(self, x):
        cnn_feat = self.cnn(x)
        cnn_feat_pooled = self.avgpool(cnn_feat).squeeze(-1).squeeze(-1)

        vit_feat = self.transformer(x)
        combined = torch.cat((cnn_feat_pooled, vit_feat), dim=1)

        return self.fc(combined)



In [20]:
#  Train function
def train_model(model, epochs=10,save_path="best_model.pth"):
    model.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=1e-4)

    for epoch in range(epochs):
        total_loss, correct, total = 0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        val_acc = validate_model(model)
        print(f"Epoch {epoch+1}/{epochs} - Train Acc: {100*correct/total:.2f}%, Val Acc: {val_acc:.2f}%")


In [16]:
#  Validation function
def validate_model(model):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    model.train()
    return 100 * correct / total


In [17]:
#  Evaluation function
def evaluate_model(model):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    print("\nConfusion Matrix:")
    print(confusion_matrix(all_labels, all_preds))
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=["normal", "bacterial", "viral"]))


In [18]:
model = ResFormer().to(device)
train_model(model, epochs=10)

Epoch 1/10 - Train Acc: 75.75%, Val Acc: 85.21%
Epoch 2/10 - Train Acc: 84.46%, Val Acc: 85.34%
Epoch 3/10 - Train Acc: 86.36%, Val Acc: 87.09%
Epoch 4/10 - Train Acc: 87.72%, Val Acc: 86.09%
Epoch 5/10 - Train Acc: 88.41%, Val Acc: 85.84%
Epoch 6/10 - Train Acc: 89.55%, Val Acc: 79.95%
Epoch 7/10 - Train Acc: 89.91%, Val Acc: 88.35%
Epoch 8/10 - Train Acc: 91.58%, Val Acc: 87.09%
Epoch 9/10 - Train Acc: 92.06%, Val Acc: 86.47%
Epoch 10/10 - Train Acc: 92.96%, Val Acc: 89.10%


In [19]:
print("\nTesting Accuracy:")
evaluate_model(model)


Testing Accuracy:

Confusion Matrix:
[[499   3  15]
 [  3 466  87]
 [  0  93 430]]

Classification Report:
              precision    recall  f1-score   support

      normal       0.99      0.97      0.98       517
   bacterial       0.83      0.84      0.83       556
       viral       0.81      0.82      0.82       523

    accuracy                           0.87      1596
   macro avg       0.88      0.88      0.88      1596
weighted avg       0.88      0.87      0.87      1596

